# Batch Download of DraCor Data with a Provenance Record

> **Author:** Ingo Börner (University of Potsdam, Germany), <a href="https://orcid.org/0000-0001-8294-2541"><img src="https://orcid.org/assets/vectors/orcid.logo.icon.svg" alt="ORCID iD" width="16" height="16" style="vertical-align: -2px"/> 0000-0001-8294-2541</a>
> **Contributors:** Frank Fischer (Freie Universität Berlin, Germany), <a href="https://orcid.org/0000-0003-2419-6629"><img src="https://orcid.org/assets/vectors/orcid.logo.icon.svg" alt="ORCID iD" width="16" height="16" style="vertical-align: -2px"/> 0000-0003-2419-6629</a>: idea ([#2](https://github.com/dracor-org/dracor-notebooks/issues/2), 2021)
> **Version:** 1 – DraCor API v1, PyDraCor 3 (November 2026); see the version history at the end of the notebook
> **Tested with:** DraCor API 1.4.1 (production server) and 1.5.0-rc.1 (staging server), PyDraCor 3.0.2, Python 3.10 to 3.12, 2026-11-17

This notebook downloads data of many plays at once: for a whole corpus, or for a selection of plays. It offers the same data as the **Downloads** tab on the page of a play on dracor.org (see e.g. [*Emilia Galotti*](https://dracor.org/ger/lessing-emilia-galotti#downloads)): networks as CSV, GEXF or GraphML, lists of characters, spoken text, stage directions, and the full text as TEI.

Along with the data, the notebook writes a **provenance record**. It records the DraCor ID of each play and the version (Git commit) of the corpus the data comes from. With these, the download can be reproduced later, and a study based on it can say exactly which data it used.

The notebook uses [PyDraCor](https://pypi.org/project/pydracor/), the Python package for the DraCor API. For an introduction to the API itself, see chapter 5 of the [DraCor Textbook](https://dracor-org.github.io/dracor-textbook/chapters/05-api-working-with-dracor-programmatically.html).

## Setting up the notebook

Install the required packages with `pip install -r requirements.txt`, or uncomment and run the following cell.

In [1]:
# !pip install "pydracor>=3.0" pandas

In [2]:
import hashlib
import io
import json
import platform
import sys
import time
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import pandas as pd
from pydracor import DraCorAPI
from pydracor.api_wrapper import DownloadFormat

The DraCor server is set in the variable `api_base`. To use the staging server, change it to `"https://staging.dracor.org/api/v1"`; for a DraCor instance running on your own computer, e.g. to `"http://localhost:8088/api/v1"`.

In [3]:
# Base URL of the DraCor API used in this notebook
api_base = "https://dracor.org/api/v1"

api = DraCorAPI(host=api_base)
api_info = api.get_info()
print(f"{api_info.name} {api_info.version} at {api_info.base}")

DraCor API v1 1.4.1 at https://dracor.org/api/v1


## What can be downloaded

The following table lists the data offered for download. The keys follow the file names that dracor.org uses on the Downloads tab: a file is named after the DraCor ID and the name of the play, followed by the key, e.g. `ger000088-lessing-emilia-galotti.network.gexf`.

Each entry names the method of PyDraCor that fetches the data, and the path of the API endpoint behind it. The path is only used to record the request in the provenance record.

In [4]:
def to_json(models):
    """Serialise a list of PyDraCor models to JSON, with the field names of the API."""
    return json.dumps([m.model_dump(mode="json", by_alias=True, exclude_none=True) for m in models],
                      ensure_ascii=False, indent=2)


# key: (description, function that fetches the data of a play, path of the API endpoint)
DOWNLOAD_TYPES = {
    "network.csv": ("Co-presence network (CSV)", lambda p, f: p.get_networkdata(DownloadFormat.csv), "networkdata/csv"),
    "network.gexf": ("Co-presence network (GEXF)", lambda p, f: p.get_networkdata(DownloadFormat.gexf), "networkdata/gexf"),
    "network.graphml": ("Co-presence network (GraphML)", lambda p, f: p.get_networkdata(DownloadFormat.graphml), "networkdata/graphml"),
    "relations.csv": ("Relations (CSV)", lambda p, f: p.get_relations(DownloadFormat.csv), "relations/csv"),
    "relations.gexf": ("Relations (GEXF)", lambda p, f: p.get_relations(DownloadFormat.gexf), "relations/gexf"),
    "relations.graphml": ("Relations (GraphML)", lambda p, f: p.get_relations(DownloadFormat.graphml), "relations/graphml"),
    "characters.csv": ("List of characters with metrics (CSV)", lambda p, f: p.get_characters_csv(), "characters/csv"),
    "characters.json": ("List of characters with metrics (JSON)", lambda p, f: to_json(p.get_characters()), "characters"),
    "spoken-text.txt": ("Spoken text, optionally filtered (TXT)", lambda p, f: p.get_spoken_text(**f), "spoken-text"),
    "spoken-text-by-character.json": ("Spoken text by character (JSON)", lambda p, f: to_json(p.get_spoken_text_by_character()), "spoken-text-by-character"),
    "stage-directions.txt": ("Stage directions (TXT)", lambda p, f: p.get_stage_directions(), "stage-directions"),
    "stage-directions-with-speakers.txt": ("Stage directions and speakers (TXT)", lambda p, f: p.get_stage_directions_with_speakers(), "stage-directions-with-speakers"),
    "tei.xml": ("Full text (TEI)", lambda p, f: p.get_tei(), "tei"),
    "txt": ("Full text (plain text)", lambda p, f: p.get_txt(), "txt"),
}

pd.DataFrame([(k, v[0], v[2]) for k, v in DOWNLOAD_TYPES.items()],
             columns=["key", "data", "API endpoint (…/plays/{playname}/…)"])

,key,data,API endpoint (…/plays/{playname}/…)
0,network.csv,Co-presence network (CSV),networkdata/csv
1,network.gexf,Co-presence network (GEXF),networkdata/gexf
2,network.graphml,Co-presence network (GraphML),networkdata/graphml
3,relations.csv,Relations (CSV),relations/csv
4,relations.gexf,Relations (GEXF),relations/gexf
5,relations.graphml,Relations (GraphML),relations/graphml
6,characters.csv,List of characters with metrics (CSV),characters/csv
7,characters.json,List of characters with metrics (JSON),characters
8,spoken-text.txt,"Spoken text, optionally filtered (TXT)",spoken-text
9,spoken-text-by-character.json,Spoken text by character (JSON),spoken-text-by-character


Relations (e.g. kinship or love relations between characters) are only encoded for some plays, mainly in GerDraCor. For plays without relations, these files are skipped.

The spoken text can be filtered by the characters' sex (`MALE`, `FEMALE`, `UNKNOWN`), by role, or by relation. The filter applies only to `spoken-text.txt`.

## The download function

The function `batch_download` downloads the requested data for the selected plays of a corpus, writes one file per play and type of data to a folder, and adds two files:

* `metadata.csv`: the metadata of the downloaded plays, as provided by the API for the whole corpus;
* `provenance.json`: the provenance record (see below).

To be gentle with the server, the function pauses briefly after each play.

In [5]:
def batch_download(corpus_name, downloads, play_names=None, spoken_text_filter=None,
                   output_dir="downloads", pause=0.5):
    """Download data of the plays of a corpus and write a provenance record.

    corpus_name: name of the corpus, e.g. "ger"
    downloads: list of keys of DOWNLOAD_TYPES, e.g. ["network.gexf"]
    play_names: list of play names; None downloads all plays of the corpus
    spoken_text_filter: filter for "spoken-text.txt", e.g. {"sex": "FEMALE"}
    output_dir: folder for the downloaded files
    pause: seconds to wait after each play
    """
    unknown = [d for d in downloads if d not in DOWNLOAD_TYPES]
    if unknown:
        raise ValueError(f"Unknown download type(s): {unknown}")
    spoken_text_filter = {k: v for k, v in (spoken_text_filter or {}).items() if v}

    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    retrieved = datetime.now(timezone.utc).isoformat(timespec="seconds")

    corpus = api.get_corpus(corpus_name)
    names = play_names or [p.name for p in sorted(corpus.plays, key=lambda p: p.id)]

    plays = []
    for name in names:
        play = corpus.get_play(name)
        if play is None:
            print(f"Play not found in {corpus_name}: {name}")
            continue
        files = []
        for key in downloads:
            if key.startswith("relations.") and not play.relations:
                files.append({"type": key, "skipped": "no relations encoded in this play"})
                continue
            _, fetch, path = DOWNLOAD_TYPES[key]
            content = fetch(play, spoken_text_filter)
            filename = f"{play.id}-{play.name}.{key}"
            (out / filename).write_text(content, encoding="utf-8")
            url = f"{api_base}/corpora/{corpus.name}/plays/{play.name}/{path}"
            if key == "spoken-text.txt" and spoken_text_filter:
                url += "?" + "&".join(f"{k}={v}" for k, v in spoken_text_filter.items())
            files.append({"type": key, "file": filename, "url": url,
                          "sha256": hashlib.sha256(content.encode("utf-8")).hexdigest()})
        plays.append({"id": play.id, "name": play.name, "title": play.title,
                      "commit": play.commit, "files": files})
        time.sleep(pause)

    # Metadata of the downloaded plays
    metadata = pd.read_csv(io.StringIO(corpus.get_metadata_csv()))
    metadata = metadata[metadata["name"].isin([p["name"] for p in plays])]
    metadata.to_csv(out / "metadata.csv", index=False)

    record = provenance_record(corpus, plays, downloads, spoken_text_filter, retrieved)
    (out / "provenance.json").write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")

    n_files = sum(1 for p in plays for f in p["files"] if "file" in f)
    print(f"{n_files} files written to {out}/ (plays: {len(plays)}), with metadata.csv and provenance.json")
    return record

## The provenance record

DraCor corpora are *living corpora*: plays are added, and existing plays are corrected. A play's text, its list of characters and its network can therefore differ between two downloads. In their report on versioning, Börner and Trilcke show how hard it is to identify the data a study was based on when a corpus is cited only by its name and the number of its plays, and they recommend citing the version of the corpus, i.e. its Git commit (Börner and Trilcke 2024, ch. 3 and 5). Since API version 1.1.0, the DraCor API returns the commit of each corpus (Börner et al. 2025, ch. 3.2).

The provenance record is the user's side of this: it writes down the state of the data at the moment of the download. Its core are two values:

* the **DraCor ID** of each play, e.g. `ger000088`, which identifies the play independently of its name and of the server, and
* the **commit** (Git SHA) of the corpus version the data comes from.

With these two, anyone can get exactly the same plays again. In addition, the record states when the data was retrieved, from which server and API version, with which client, and which data and filters were requested. For each file, it gives the request it came from and a checksum (SHA-256), so that it can be checked later whether a file was changed.

In [6]:
def provenance_record(corpus, plays, downloads, spoken_text_filter, retrieved):
    """Assemble the provenance record of a download."""
    ids = ", ".join(p["id"] for p in plays)
    citation = (f"{corpus.title} ({corpus.acronym}), {corpus.repository}, commit {corpus.commit}. "
                f"Licence: {corpus.licence}. Retrieved {retrieved[:10]} via {api_info.name} {api_info.version} "
                f"({api_base}). Plays: {ids}.")
    return {
        "retrieved": retrieved,
        "corpus": {
            "name": corpus.name,
            "title": corpus.title,
            "acronym": corpus.acronym,
            "repository": corpus.repository,
            "commit": corpus.commit,
            "licence": corpus.licence,
            "licenceUrl": corpus.licence_url,
        },
        "api": api_info.model_dump(mode="json", by_alias=True, exclude_none=True),
        "client": {"pydracor": version("pydracor"), "python": platform.python_version()},
        "request": {"downloads": downloads, "spokenTextFilter": spoken_text_filter},
        "plays": plays,
        "citation": citation,
    }

## Example 1: GEXF networks of a few plays

A typical task: the co-presence networks of a few plays as GEXF files, e.g. to analyse and visualise them in [Gephi](https://gephi.org). Here we download the networks of three plays of the German Drama Corpus (GerDraCor), together with their lists of characters.

In [7]:
record = batch_download(
    corpus_name="ger",
    play_names=["lessing-emilia-galotti", "schiller-kabale-und-liebe", "goethe-torquato-tasso"],
    downloads=["network.gexf", "characters.csv"],
    output_dir="downloads/ger-networks",
)

6 files written to downloads/ger-networks/ (plays: 3), with metadata.csv and provenance.json


The provenance record of this download:

In [8]:
print(json.dumps(record, ensure_ascii=False, indent=2))

{
  "retrieved": "2026-11-16T23:59:25+00:00",
  "corpus": {
    "name": "ger",
    "title": "German Drama Corpus",
    "acronym": "GerDraCor",
    "repository": "https://github.com/dracor-org/gerdracor",
    "commit": "4919ac26811d2cde453de6d3ef22d0b6689a3bb1",
    "licence": "CC0",
    "licenceUrl": "https://creativecommons.org/share-your-work/public-domain/cc0/"
  },
  "api": {
    "name": "DraCor API v1",
    "version": "1.4.1",
    "status": "stable",
    "existdb": "6.4.0",
    "base": "https://dracor.org/api/v1",
    "openapi": "https://dracor.org/api/v1/openapi.yaml"
  },
  "client": {
    "pydracor": "3.0.2",
    "python": "3.11.17"
  },
  "request": {
    "downloads": [
      "network.gexf",
      "characters.csv"
    ],
    "spokenTextFilter": {}
  },
  "plays": [
    {
      "id": "ger000088",
      "name": "lessing-emilia-galotti",
      "title": "Emilia Galotti",
      "commit": "4919ac26811d2cde453de6d3ef22d0b6689a3bb1",
      "files": [
        {
          "type": "netwo

The record ends with a citation suggestion, which can be used to cite the data in a publication:

In [9]:
print(record["citation"])

German Drama Corpus (GerDraCor), https://github.com/dracor-org/gerdracor, commit 4919ac26811d2cde453de6d3ef22d0b6689a3bb1. Licence: CC0. Retrieved 2026-11-16 via DraCor API v1 1.4.1 (https://dracor.org/api/v1). Plays: ger000088, ger000096, ger000082.


## Example 2: Spoken text of female characters in a whole corpus

The original idea for this notebook ([#2](https://github.com/dracor-org/dracor-notebooks/issues/2)) was to download the text spoken by female characters for all plays of a corpus, e.g. to compare it with the text of male characters in a tool such as [Voyant](https://voyant-tools.org) or [stylo](https://github.com/computationalstylistics/stylo). Here we do this for the German Shakespeare Drama Corpus (GerShDraCor, 38 plays).

In [10]:
record_gersh = batch_download(
    corpus_name="gersh",
    downloads=["spoken-text.txt"],
    spoken_text_filter={"sex": "FEMALE"},
    output_dir="downloads/gersh-spoken-text-female",
)
print(record_gersh["citation"])

38 files written to downloads/gersh-spoken-text-female/ (plays: 38), with metadata.csv and provenance.json
German Shakespeare Drama Corpus (GerShDraCor), https://github.com/dracor-org/gershdracor, commit bf7f0a8a4efd27f9779ff159871bbcf0895ab90c. Licence: CC0. Retrieved 2026-11-16 via DraCor API v1 1.4.1 (https://dracor.org/api/v1). Plays: gersh000001, gersh000002, gersh000003, gersh000004, gersh000005, gersh000006, gersh000007, gersh000008, gersh000009, gersh000010, gersh000011, gersh000012, gersh000013, gersh000014, gersh000015, gersh000016, gersh000017, gersh000018, gersh000019, gersh000020, gersh000021, gersh000022, gersh000023, gersh000024, gersh000025, gersh000026, gersh000027, gersh000028, gersh000029, gersh000030, gersh000031, gersh000032, gersh000033, gersh000034, gersh000035, gersh000036, gersh000037, gersh000038.


## Checking a download later

Because the provenance record stores the commit of the corpus, it can be checked later whether the corpus on the server has changed since the download. The function `check_provenance` compares the commit in a record with the current commit of the corpus and of each play.

In [11]:
def check_provenance(path):
    """Compare a provenance record with the current state of the corpus on the server."""
    record = json.loads(Path(path).read_text(encoding="utf-8"))
    corpus = api.get_corpus(record["corpus"]["name"])
    then, now = record["corpus"]["commit"], corpus.commit
    print(f"Corpus {corpus.name}: commit at download {then[:7]}, now {now[:7]}")
    if then == now:
        print("The corpus is unchanged since the download.")
        return
    print("The corpus has changed since the download. Plays with a different commit now:")
    for p in record["plays"]:
        current = corpus.get_play(p["name"])
        if current is None:
            print(f"  {p['id']} {p['name']}: no longer in the corpus")
        elif current.commit != p["commit"]:
            print(f"  {p['id']} {p['name']}: {p['commit'][:7]} -> {current.commit[:7]}")
    print(f"The corpus at the time of the download: {record['corpus']['repository']}/tree/{then}")

In [12]:
check_provenance("downloads/ger-networks/provenance.json")

Corpus ger: commit at download 4919ac2, now 4919ac2
The corpus is unchanged since the download.


The link at the end of the output (if the corpus has changed) leads to the state of the corpus repository at the recorded commit. There, the TEI files of the plays can be retrieved in exactly the version that was downloaded.

## Using the notebook as a script

The functions of this notebook can also be used as a Python script, e.g. to run downloads on a server or regularly. The following cell defines a command-line interface. In the notebook, it does nothing; it only runs when the code is executed as a script.

In [13]:
def main(argv=None):
    """Command-line interface: python batch_download.py CORPUS TYPE [TYPE ...] [options]"""
    import argparse

    global api, api_base, api_info
    parser = argparse.ArgumentParser(description="Batch download of DraCor data with a provenance record.")
    parser.add_argument("corpus", help="name of the corpus, e.g. ger")
    parser.add_argument("downloads", nargs="+", choices=list(DOWNLOAD_TYPES), metavar="TYPE",
                        help="types of data: " + ", ".join(DOWNLOAD_TYPES))
    parser.add_argument("--plays", nargs="+", help="names of plays (default: all plays of the corpus)")
    parser.add_argument("--sex", help="filter the spoken text by sex: MALE, FEMALE or UNKNOWN")
    parser.add_argument("--output", default="downloads", help="folder for the files (default: downloads)")
    parser.add_argument("--api-base", default=api_base, help=f"DraCor API (default: {api_base})")
    args = parser.parse_args(argv)

    if args.api_base != api_base:
        api_base = args.api_base
        api = DraCorAPI(host=api_base)
        api_info = api.get_info()
    batch_download(args.corpus, args.downloads, play_names=args.plays,
                   spoken_text_filter={"sex": args.sex}, output_dir=args.output)


# Runs only when this code is executed as a script, not in Jupyter
if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

To create the script, convert the notebook with [nbconvert](https://nbconvert.readthedocs.io), which is part of Jupyter. The cells with the examples are tagged `example` and are left out:

```sh
jupyter nbconvert --to script --TagRemovePreprocessor.enabled=True \
    --TagRemovePreprocessor.remove_cell_tags example batch_download.ipynb
```

This writes the file `batch_download.py`. Example 1 then becomes:

```sh
python batch_download.py ger network.gexf characters.csv \
    --plays lessing-emilia-galotti schiller-kabale-und-liebe goethe-torquato-tasso \
    --output downloads/ger-networks
```

and Example 2:

```sh
python batch_download.py gersh spoken-text.txt --sex FEMALE --output downloads/gersh-spoken-text-female
```

`python batch_download.py --help` lists all options. Since the script is created from the notebook, it changes with it: after an update of the notebook, convert it again.

## Redoing an export

The DraCor IDs of the plays, the commit of the corpus and the version of the API together describe the state of the data behind a download. Since DraCor publishes a Docker image for every release of the API (Börner et al. 2025, ch. 3.2), this state can in principle be rebuilt on one's own computer: start the API image of the recorded version, load the corpus at the recorded commit, and run this notebook against the local instance (set `api_base` accordingly). The export is then repeated on exactly the same data. This follows the workflow described in CLS INFRA D7.3, which recommends container technology "to conserve the state of data and code at the time when conducting the research" (Börner and Trilcke 2024, ch. 4.2 and 5).

For a hands-on introduction to running DraCor with Docker, see the section "Setting up a local DraCor Environment with Docker" in the notebook [reproducible-research-with-docker](https://github.com/dracor-org/dracor-notebooks/tree/main/reproducible-research-with-docker).

Two limitations:

* Only API versions since 1.1.0 return the commit of the corpus, so the record can only be filled for downloads from these versions.
* The network metrics in the downloads (e.g. in the lists of characters) are computed by a separate metrics service, whose version is not reported by the API. The record covers the data and the API, but not this service.

## References

Börner, Ingo, and Peer Trilcke. *CLS INFRA D7.3 On Versioning Living and Programmable Corpora*. Zenodo, 2024. [https://doi.org/10.5281/zenodo.11081934](https://doi.org/10.5281/zenodo.11081934).

Börner, Ingo, Peer Trilcke, Luca Giovannini, and Daniil Skorinkin. *CLS INFRA D7.4 Report on the Implementation of Programmable Corpora*. Zenodo, 2025. [https://doi.org/10.5281/zenodo.15301341](https://doi.org/10.5281/zenodo.15301341).

## Use of AI

This notebook was drafted with Claude Code (Anthropic), based on the proposal in [#2](https://github.com/dracor-org/dracor-notebooks/issues/2) and the instructions of the author, and reviewed by the author.

## Version history

The version history of this notebook is documented in [README.md](https://github.com/dracor-org/dracor-notebooks/blob/main/batch_download/README.md). In the [DraCor Notebooks book](https://dracor-org.github.io/dracor-notebooks/), it follows below.

```{include} README.md
:start-after: "## Version history"
```